Dans cette dernière partie, nous souhaitons mettre en place en clustering.

In [ ]:
%%capture
%run script_nettoyage.ipynb

1. Préparation des variables

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import random

random.seed(1)

# copie
df = df_freq_totale.copy()

# colonnes non binaires
colonnes_non_binaires = [
    'REF DU MUSEE', 'NOMREG', 'NOM DU MUSEE', 'VILLE',
    'Fréquentation', 'IDMuseofile', 'annee', 'frequentation',
    'speciale', 'freq_net', 'Statut', 'Domaine_thematique'
]

# colonnes binaires thématiques
colonnes_binaires = [col for col in df.columns if col not in colonnes_non_binaires]

# types de variables
variables_numeriques = ['freq_net']
variables_categorielles = ['annee', 'NOMREG']

# conversions
df['freq_net'] = pd.to_numeric(df['freq_net'], errors='coerce')

# sous-table
X = df[variables_numeriques + variables_categorielles + colonnes_binaires].copy()

# Gestion des NA :
for col in variables_numeriques:
    X[col] = X[col].fillna(X[col].median())

for col in variables_categorielles:
    X[col] = X[col].fillna("Inconnu").astype(str)

for col in colonnes_binaires:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)

# prétraitement. La variable numérique est standardisée, les variables catégorielles sont transformées en un ensemble de variables binaires
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), variables_numeriques),
        ('cat', OneHotEncoder(handle_unknown='ignore'), variables_categorielles),
        ('bin', 'passthrough', colonnes_binaires)
    ]
)

X_prep = preprocessor.fit_transform(X)

2. Choix du nombre de clusters

In [ ]:
inerties = []
K = range(1, 11)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_prep)
    inerties.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K, inerties, marker='o')
plt.xlabel("Nombre de clusters k")
plt.ylabel("Inertie")
plt.title("Méthode du coude")
plt.xticks(K)
plt.show()

La "règle du coude" nous permet de choisir le nombre de clusters. Ici nous en choisissons 3.

3. Clustering

In [ ]:
# clustering final
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=20)
df['cluster'] = kmeans_final.fit_predict(X_prep)

# réduction en 2 dimensions
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_prep)

# graphique
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['cluster'])
plt.xlabel("Composante principale 1")
plt.ylabel("Composante principale 2")
plt.title("Représentation des clusters")
plt.colorbar(scatter, label="Cluster")
plt.show()

print(df[['NOM DU MUSEE', 'annee', 'NOMREG', 'Statut', 'freq_net', 'cluster']].head())

Un cluster semble se distinguer des autres par rapport à la composante principale 1 et les deux autres se différencient sur la composante 2.

In [ ]:
score_silhouette = silhouette_score(X_prep, df['cluster'])
print("Score de silhouette :", score_silhouette)

Le score silhouette est très faible ce qui indique que les clusters ne sont pas très bien distincts.

Combien d'observations sont présentes dans chaque cluster ?

In [ ]:
df['cluster'].value_counts().sort_index()

Nous remarquons que le cluster 2 est beaucoup plus petit que les 2 autres.

Nous souhaitons identifier quelles variables contribuent le plus aux deux premières composantes principales de la PCA.

In [ ]:
# PCA sur les données préparées
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_prep)

# Noms des variables après prétraitement
feature_names = preprocessor.get_feature_names_out()

# loadings = contribution des variables aux composantes
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_names,
    columns=['PC1', 'PC2']
)

top_pc1 = loadings.sort_values(by='PC1', key=lambda x: x.abs(), ascending=False)
top_pc2 = loadings.sort_values(by='PC2', key=lambda x: x.abs(), ascending=False)

print("Top 10 PC1")
print(top_pc1[['PC1']].head(10))

print("\nTop 10 PC2")
print(top_pc2[['PC2']].head(10))

La composante principale 1 est portée par la variable de fréquence. Les observations présentes dans le cluster portée par cette composante pourraient donc présenter des fréquentations très importantes.

In [ ]:
resume_num = df.groupby('cluster').agg({
    'freq_net': ['mean', 'median']
})

print(resume_num)

L'hypothèse est bien vérifiée. Le cluster 2 est clairement composé d'observations ayant une fréquentation plus importante que les autres.

Les clusters 0 et 1 pourraient se distinguer par rapport aux domaines thématiques. Vérifions cette hypothèse.

In [ ]:
# Moyenne des thématiques pour les clusters 0 et 1
themes_01 = df[df['cluster'].isin([0, 1])] \
    .groupby('cluster')[colonnes_binaires] \
    .mean() \
    .T

# Ne garder que les thématiques les plus présentes au total
themes_01['moyenne_totale'] = themes_01.mean(axis=1)
themes_01 = themes_01.sort_values('moyenne_totale', ascending=False).head(20)
themes_01 = themes_01.drop(columns='moyenne_totale')

ax = themes_01.plot(kind='barh', figsize=(10, 8))

plt.xlabel("Proportion dans le cluster")
plt.ylabel("Thématique")
plt.title("Comparaison des thématiques binaires : cluster 0 vs cluster 1")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

Le cluster 1 se distingue effectivement par les beaux-arts et arts en général. Le cluster 0 semble se distinguer par la technologie et la science.

L'année ne semble pas avoir d'influence sur le clustering. Vérifions cela.

In [ ]:
table_annee_cluster = pd.crosstab(df['annee'], df['cluster'])
print(table_annee_cluster)

La répartition des clusters selon l'année est effectivement très stable.

Nous nous demandons maintenant si les clusters 0 et 1 se différencient également par les régions. Nous commençons par regarder quelles régions sont plutôt présentes dans le cluster 0 et 1.

In [ ]:
df_01 = df[df['cluster'].isin([0, 1])].copy()

table_region = pd.crosstab(
    df_01['NOMREG'],
    df_01['cluster'],
    normalize='index'
)

if 0 not in table_region.columns:
    table_region[0] = 0
if 1 not in table_region.columns:
    table_region[1] = 0

table_region['diff_0_1'] = table_region[0] - table_region[1]
table_region = table_region.sort_values('diff_0_1')

plt.figure(figsize=(10, 8))
plt.barh(table_region.index, table_region['diff_0_1'])
plt.xlabel("Part cluster 0 - part cluster 1")
plt.ylabel("Région")
plt.title("Dominance du cluster 0 vs cluster 1 selon la région")
plt.show()

Les régions avec très peu de musées (Saint-Pierre et Miquelon, Gyuane etc.) sont particulièrement présentes dans le cluster 0.

Nous souhaitons savoir si les régions plutôt présentes dans le cluster 0 sont également associées à des thématiques scientifiques et de même avec le cluster 1 et les thématiques artistiques. Pour cela nous gardons les 4 régions les plus présentes dans chaque cluster.

In [ ]:
regions_c0 = table_region.sort_values('diff_0_1', ascending=False).head(4).index
regions_c1 = table_region.sort_values('diff_0_1', ascending=True).head(4).index

print("Régions plutôt cluster 0 :", regions_c0)
print("Régions plutôt cluster 1 :", regions_c1)

Nous regardons ensuite les thématiques les plus fréquentes dans chaque cluster avec ces régions.

In [ ]:
colonnes_non_binaires = [
    'REF DU MUSEE', 'NOMREG', 'NOM DU MUSEE', 'VILLE',
    'Fréquentation', 'IDMuseofile', 'annee', 'frequentation',
    'speciale', 'freq_net', 'Statut', 'Domaine_thematique', 'cluster'
]

colonnes_binaires = [col for col in df.columns if col not in colonnes_non_binaires]

themes_regions_c0 = df[df['NOMREG'].isin(regions_c0)][colonnes_binaires].mean().sort_values(ascending=False)
print(themes_regions_c0.head(10))

themes_regions_c1 = df[df['NOMREG'].isin(regions_c1)][colonnes_binaires].mean().sort_values(ascending=False)
print(themes_regions_c1.head(15))

comparaison_themes = pd.DataFrame({
    'regions_plutot_cluster0': df[df['NOMREG'].isin(regions_c0)][colonnes_binaires].mean(),
    'regions_plutot_cluster1': df[df['NOMREG'].isin(regions_c1)][colonnes_binaires].mean()
})

comparaison_themes['ecart'] = (
    comparaison_themes['regions_plutot_cluster0'] -
    comparaison_themes['regions_plutot_cluster1']
)

comparaison_themes = comparaison_themes.sort_values('ecart', ascending=False)

print(comparaison_themes.head(15))

Nous retrouvons les tendances montrées précédemment. Les régions les plus associées à chaque cluster ont différent profils thématiques. Cependant ce résultat est à prendre avec du recul car les 4 régions sélectionées pour le cluster 0 sont des régions avec très peu de musées.

## Conclusion du clustering

L’analyse par clustering met en évidence une structuration en trois groupes, même si leur séparation reste limitée, comme l’indique le faible score de silhouette. Les clusters obtenus ne constituent donc pas des classes parfaitement distinctes, mais plutôt des profils descriptifs. 
Un premier résultat est l’existence d’un petit cluster très spécifique, le cluster 2, nettement isolé sur la première composante principale. Cette composante étant essentiellement portée par la fréquentation, ce cluster correspond à des observations caractérisées par des niveaux de fréquentation très élevés. Il s’agit donc d’un groupe atypique, minoritaire, mais très marqué.
Les deux autres clusters, qui concentrent l’essentiel des observations, se différencient avant tout sur la seconde composante principale, davantage liée aux domaines thématiques. Le cluster 1 apparaît associé aux beaux-arts et plus largement aux thématiques artistiques, tandis que le cluster 0 se rapproche davantage de profils liés à la technologie et aux sciences.
Cette lecture est renforcée par l’analyse temporelle, qui montre une très forte stabilité de la répartition des clusters selon l’année. L’année n’apparaît donc pas comme un facteur structurant majeur de la classification.
Finalement, certaines régions ayant peu de musées sont davantage représentées dans l’un ou l’autre des clusters 0 et 1. L’analyse des thématiques dans les régions les plus représentées au sein de chaque cluster retrouve globalement les tendances déjà observées, avec d’un côté des profils davantage scientifiques et techniques, de l’autre des profils plus artistiques. Cependant la dimension régionale ne semble pas avoir beaucoup d'influence sur la classification.